In [1]:
import sys
import warnings
import numpy as np
import pathlib as pl
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern
from scipy.stats import norm
from scipy.optimize import minimize
root_folder = pl.Path.cwd().parents[2]
sys.path.insert(0, str(root_folder / "utilities"))
import common_functions as cf

warnings.filterwarnings('ignore')

initial_data_folder = "data/initial_data/function_5"
initial_inputs_path = pl.Path.joinpath(root_folder, initial_data_folder,  "initial_inputs.npy")
initial_outputs_path = pl.Path.joinpath(root_folder, initial_data_folder, "initial_outputs.npy")

In [2]:
data_in = np.load(initial_inputs_path)
data_out = np.load(initial_outputs_path)

Week-01

In [3]:
kernel = Matern(nu=2.5)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, normalize_y=True)
gp.fit(data_in, data_out)

# Surrogate to maximise (negative for minimize)
def surrogate_neg(x):
    return -gp.predict(x.reshape(1, -1))[0]

# Bounds for normalized inputs
bounds = [(0,1), (0,1), (0,1), (0,1)]

# Try multiple random starts to avoid local issues
best_x = None
best_val = float('inf')
for _ in range(10):
    x0 = np.random.rand(4)
    res = minimize(surrogate_neg, x0=x0, bounds=bounds, method='L-BFGS-B')
    if res.fun < best_val:
        best_val = res.fun
        best_x = res.x

print(cf.format_inputdata(best_x))

0.976559-0.107850-0.831282-0.161520


In [4]:
# --- Step 3: Define Expected Improvement acquisition function ---
def expected_improvement(x, gp, y_best, xi=0.01):
    x = np.array(x).reshape(1, -1)
    mu, sigma = gp.predict(x, return_std=True)
    mu, sigma = mu[0], sigma[0]
    if sigma == 0.0:
        return 0.0
    imp = mu - y_best - xi
    Z = imp / sigma
    ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
    return -ei  # negative because we will minimize

Week-02

In [5]:
new_points = np.array([
    [0.232877,0.841416,0.883342,0.879464]
])

new_outputs = np.array([
    1091.3271430129832
])


# Combine all data
X_all = np.vstack([data_in, new_points])
y_all = np.concatenate([data_out, new_outputs])

# --- Refit Gaussian Process ---
kernel = Matern(nu=2.5)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, normalize_y=True)
gp.fit(X_all, y_all)

# --- Generate candidate next points around current best ---
best_x = new_points
sigma = 0.02  # tweak size for local exploration
num_candidates = 5

x_next_candidates = best_x + np.random.normal(0, sigma, size=(num_candidates, 4))
# Ensure all points are within [0,1]
x_next_candidates = np.clip(x_next_candidates, 0, 1)

print("Candidate next points to evaluate:")
print(x_next_candidates)

Candidate next points to evaluate:
[[0.22674852 0.88150635 0.87989952 0.88118554]
 [0.26996843 0.83516968 0.87596641 0.8967214 ]
 [0.25242368 0.8643373  0.85947914 0.87013781]
 [0.24314825 0.85643202 0.86811398 0.89108576]
 [0.22537681 0.86301315 0.88402389 0.90337709]]
